# Servidor Unificado en Google Colab: Inferencia (M1 + M2 + M3) + Agente Ollama (`qwen3:1.7b`)
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio 3: Sistema de Control de Peaje

Combina el patrón de `week_2_servidor_pose_colab_v2.ipynb` y `colab_server_v2.ipynb`:
1. Clona automáticamente tu repositorio `https://github.com/jhairp/Peaje.git`.
2. Instala y levanta **Ollama** con el modelo **`qwen3:1.7b`** en la GPU de Colab (sin necesidad de API Keys de pago externas).
3. Levanta el servidor FastAPI con los **3 modelos de visión (`M1 Placas+OCR`, `M2 Segmentación`, `M3 Keypoints Ejes`)** + **`/llm-chat` (Ollama Qwen)**.
4. Expone todo en una única URL pública HTTPS con **Cloudflare Tunnel (`trycloudflare.com`)** para pegar en `INFERENCE_URL` de Render.

In [ ]:
# ── 1. Parámetros ────────────────────────────────────────────────
import os

REPO        = 'https://github.com/jhairp/Peaje.git'
BRANCH      = 'main'
OLLAMA_MODEL = 'qwen3:1.7b'
PUERTO      = 8000
API_TOKEN   = 'token_secreto_inferencia_peaje_2026'

os.environ['API_TOKEN'] = API_TOKEN
os.environ['OLLAMA_MODEL'] = OLLAMA_MODEL
print('Token configurado:', API_TOKEN)

In [ ]:
# ── 2. Clonar repositorio e instalar dependencias ────────────────
%cd /content
!rm -rf Peaje && git clone -q -b $BRANCH $REPO
%cd /content/Peaje/inferencia
!pip install -q ultralytics==8.3.27 easyocr==1.7.2 fastapi==0.115.4 "uvicorn[standard]==0.32.0" python-multipart opencv-python-headless requests

In [ ]:
# ── 3. Instalar y arrancar Ollama con Qwen (igual que en colab_server_v2) ──
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/content/ollama.log', 'w'),
    stderr=subprocess.STDOUT
)
for _ in range(30):
    try:
        requests.get('http://localhost:11434/api/tags', timeout=2)
        break
    except Exception:
        time.sleep(1)
print('Ollama OK')
!ollama pull $OLLAMA_MODEL
!ollama list

In [ ]:
# ── 4. Diagnóstico del entorno en la nube (Bonificación +5 pts) ──
from diagnosticar_entorno import diagnosticar
diagnosticar()

In [ ]:
# ── 5. Iniciar Servidor FastAPI (M1+M2+M3+Qwen) y Túnel Cloudflare ──
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

import threading, re, uvicorn
from app_inferencia import app, obtener_motor

motor = obtener_motor()

if 'servidor' not in globals() or not servidor.started:
    config = uvicorn.Config(app, host='0.0.0.0', port=PUERTO, log_level='warning')
    servidor = uvicorn.Server(config)
    threading.Thread(target=servidor.run, daemon=True).start()
    time.sleep(2)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PUERTO}'],
    stdout=open('/content/tunnel.log', 'w'),
    stderr=subprocess.STDOUT
)

URL_PUBLICA = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        URL_PUBLICA = m.group(0)
        break

print('=================================================================')
print('URL PÚBLICA (Pegar en INFERENCE_URL de Render o pestaña Estado GPU):')
print(URL_PUBLICA)
print('INFERENCE_API_TOKEN:', API_TOKEN)
print('=================================================================')

In [ ]:
# ── 6. Verificar estado (/health) de M1, M2, M3 y Ollama Qwen ────
import requests
print(requests.get(f'http://localhost:{PUERTO}/health').json())